# Week 01 - Student Data Processing with Pandas & NumPy

**Objective**

Load a raw student dataset, clean it, compute summary statistics with NumPy,
classify every student as `Pass` or `Fail`, and export the cleaned dataset.

**Steps covered**

1. Load and inspect the data
2. Clean it (duplicates + missing values)
3. Compute statistics with NumPy
4. Classify students
5. Save the processed dataset


## 1. Imports and data loading

The dataset is read from `data/students.csv`. The helper below tries the local
file first, then the GitHub raw URL (useful when the notebook runs on Colab),
and finally falls back to the in-memory copy of the raw rows so the notebook
always executes end to end.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_URL = (
    "https://raw.githubusercontent.com/Tensaye-Daniel/week1-demo/main/week-01/data/students.csv"
)
FALLBACK_ROWS = [
    {"Name": "Abebe", "age": 20, "score": 85},
    {"Name": "Sara", "age": 21, "score": 90},
    {"Name": "Hana", "age": None, "score": 88},
    {"Name": "Dagim", "age": 21, "score": 99},
    {"Name": "Kiya", "age": 19, "score": 74},
    {"Name": "Biruk", "age": 20, "score": 98},
]


def load_students() -> pd.DataFrame:
    """Load students.csv from disk, from GitHub, or from the fallback rows."""
    local_path = Path("data/students.csv")
    if local_path.exists():
        return pd.read_csv(local_path)
    try:
        return pd.read_csv(RAW_URL)
    except Exception:
        print("Could not read the CSV file; using the embedded copy of the data.")
        return pd.DataFrame(FALLBACK_ROWS)


df = load_students()
print(df)

    Name   age  score
0  Abebe  20.0     85
1   Sara  21.0     90
2   Hana   NaN     88
3  Dagim  21.0     99
4   Kiya  19.0     74
5  Biruk  20.0     98


## 2. Inspecting the data

`df.info()` shows the column names, dtypes and how many missing values each
column holds. Here `age` is a `float64` with one missing value - that missing
value is what we fix in the cleaning step.

In [2]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    6 non-null      str    
 1   age     5 non-null      float64
 2   score   6 non-null      int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 276.0 bytes
None


## 3. Cleaning the data

Cleaning happens in three small, reusable functions:

- `remove_duplicates` - drops repeated student records.
- `fill_missing_values` - fills a missing **age** with the **median** (ages are
  spread out, so the median is less sensitive to outliers than the mean) and a
  missing **score** with the **mean**.
- `clean_data` - chains the two steps together.

Each function returns a new DataFrame, so the raw data is never modified in
place.

In [3]:
def remove_duplicates(data: pd.DataFrame) -> pd.DataFrame:
    """Drop fully duplicated student records."""
    cleaned = data.drop_duplicates()
    print(f"Removed {len(data) - len(cleaned)} duplicate row(s)")
    return cleaned


def fill_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    """Fill a missing age with the median age and a missing score with the mean."""
    cleaned = data.copy()
    median_age = cleaned["age"].median()
    mean_score = cleaned["score"].mean()
    cleaned["age"] = cleaned["age"].fillna(median_age)
    cleaned["score"] = cleaned["score"].fillna(mean_score)
    print(
        f"Filled missing age with median ({median_age}) "
        f"and score with mean ({round(mean_score, 2)})"
    )
    return cleaned


def clean_data(data: pd.DataFrame) -> pd.DataFrame:
    """Run the full cleaning pipeline: de-duplicate, then impute missing values."""
    return fill_missing_values(remove_duplicates(data))

Applying the pipeline and verifying that no missing values remain:

In [4]:
cleaned_df = clean_data(df)

print("\nCleaned data:")
print(cleaned_df)

print("\nRemaining missing values:")
print(cleaned_df.isnull().sum())

Removed 0 duplicate row(s)
Filled missing age with median (20.0) and score with mean (89.0)

Cleaned data:
    Name   age  score
0  Abebe  20.0     85
1   Sara  21.0     90
2   Hana  20.0     88
3  Dagim  21.0     99
4   Kiya  19.0     74
5  Biruk  20.0     98

Remaining missing values:
Name     0
age      0
score    0
dtype: int64


## 4. Statistics with NumPy

The scores are converted to a NumPy array, then `mean`, `max`, `min` and `std`
give the summary of the class. The standard deviation of about `8.41` shows the
scores sit fairly close to the average of `89`.

In [5]:
scores = cleaned_df["score"].to_numpy()

average_score = np.mean(scores)
highest_score = np.max(scores)
lowest_score = np.min(scores)
standard_deviation = np.std(scores)

print("Average Score:", round(average_score, 2))
print("Highest Score:", highest_score)
print("Lowest Score:", lowest_score)
print("Standard Deviation:", round(standard_deviation, 2))

Average Score: 89.0
Highest Score: 99
Lowest Score: 74
Standard Deviation: 8.41


## 5. Classifying students

A student **passes** when the score is 50 or above. `np.where` adds a `result`
column holding `Pass` or `Fail` for every row.

In [6]:
PASS_MARK = 50


def classify_students(data: pd.DataFrame, pass_mark: int = PASS_MARK) -> pd.DataFrame:
    """Add a `result` column marking each student as Pass or Fail."""
    classified = data.copy()
    classified["result"] = np.where(
        classified["score"] >= pass_mark,
        "Pass",
        "Fail",
    )
    return classified


final_df = classify_students(cleaned_df)

print("Final Processed Dataset:")
print(final_df)

Final Processed Dataset:
    Name   age  score result
0  Abebe  20.0     85   Pass
1   Sara  21.0     90   Pass
2   Hana  20.0     88   Pass
3  Dagim  21.0     99   Pass
4   Kiya  19.0     74   Pass
5  Biruk  20.0     98   Pass


## 6. Saving the result

In [7]:
output_path = Path("data/cleaned_students.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to {output_path}")

Cleaned dataset saved to data\cleaned_students.csv


## Summary

- Loaded `students.csv` with Pandas and inspected it with `info()`.
- Removed duplicate rows and imputed the one missing **age** with the median
  and any missing **score** with the mean - no null values remain.
- Used NumPy for the score statistics: average **89.0**, highest **99**,
  lowest **74**, standard deviation **8.41**.
- Added a `result` column that classifies every student as `Pass` or `Fail`
  using a pass mark of 50.
- Saved the processed dataset to `data/cleaned_students.csv`.

The same pipeline lives in `student_data_processing.py`, so it can be run
directly with `python student_data_processing.py`.